# Data Science: Compressed Sensing & Sparse Recovery

## LASSO, RIP, and Measurement Complexity

**Breakthrough** (Candès & Donoho, 2005): Recover k-sparse signals from m = O(k log p) measurements  
**Method**: LASSO (ℓ₁ minimization)  
**Theory**: Restricted Isometry Property (RIP)

### The Miracle
With random measurements and LASSO, we can recover exponentially sparse signals from *sublinear* measurements!


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import lstsq
from sklearn.linear_model import LassoCV, OrthogonalMatchingPursuit
import seaborn as sns

np.random.seed(42)
sns.set_style('whitegrid')

## Part 1: Theory - Restricted Isometry Property (RIP)

### Definition
A matrix A satisfies RIP with constant δₖ if:
```
(1 - δₖ)||x||₂² ≤ ||Ax||₂² ≤ (1 + δₖ)||x||₂²
for all k-sparse vectors x
```

### Guarantee (Candès-Tao)
If A satisfies RIP(2k, δ₂ₖ < 0.4), then LASSO recovers any k-sparse x exactly from:
```
min ||y - Ax||₂² + λ||x||₁
```

### Key Insight
Random Gaussian/Bernoulli matrices satisfy RIP with high probability when:
```
m ≥ C * k * log(p/k)
```


In [ ]:
def check_rip(A, k, tolerance=0.1):
    """Check if A approximately satisfies RIP(k) using eigensystem."""
    AtA = A.T @ A
    eigenvalues = np.linalg.eigvalsh(AtA)
    
    # Largest and smallest eigenvalues
    lambda_max = np.max(eigenvalues)
    lambda_min = np.min(eigenvalues[eigenvalues > 1e-10])  # Avoid zero eigenvalues
    
    # RIP constants (simplified check)
    delta_k = 1 - lambda_min
    
    return delta_k, lambda_max, lambda_min

# Generate random measurement matrix (RIP property guaranteed)
p = 500  # Signal dimension
m = 60   # Number of measurements (m << p)
k = 5    # Sparsity level

A = np.random.randn(m, p) / np.sqrt(m)  # Normalized random matrix

delta_k, lambda_max, lambda_min = check_rip(A, k)

print(f"Matrix A: {m} × {p} (underdetermined)")
print(f"Sparsity: k = {k}")
print(f"Measurement ratio: m/p = {m/p:.2%}")
print(f"Information-theoretic limit: m ≥ k*log(p/k) = {k * np.log(p/k):.0f}")
print(f"\nRIP Constant (δₖ): {delta_k:.4f} (should be < 0.4 for recovery)")

## Part 2: Sparse Signal Recovery


In [ ]:
# Generate sparse signal
x_true = np.zeros(p)
sparsity_indices = np.random.choice(p, k, replace=False)
x_true[sparsity_indices] = np.random.randn(k) * 5  # Non-zero entries

# Observe noisy measurements
noise_level = 0.01
y = A @ x_true + noise_level * np.random.randn(m)

print(f"\nTrue signal: x ∈ ℝ^{p}")
print(f"  Sparsity: {k} nonzero entries")
print(f"  Signal energy: ||x||₂ = {np.linalg.norm(x_true):.4f}")
print(f"\nMeasurements: y = Ax + ε")
print(f"  Noise level: σ = {noise_level:.4f}")
print(f"  Measurement energy: ||y||₂ = {np.linalg.norm(y):.4f}")

In [ ]:
# Recovery via LASSO
model_lasso = LassoCV(cv=5, max_iter=1000)
model_lasso.fit(A, y)
x_lasso = model_lasso.coef_

# Recovery via OMP (greedy alternative)
model_omp = OrthogonalMatchingPursuit(n_nonzero_coefs=k)
model_omp.fit(A, y)
x_omp = model_omp.coef_

# Errors
error_lasso = np.linalg.norm(x_lasso - x_true) / np.linalg.norm(x_true)
error_omp = np.linalg.norm(x_omp - x_true) / np.linalg.norm(x_true)

print("\nRECOVERY RESULTS")
print("="*60)
print(f"LASSO relative error:  {error_lasso:.4f}")
print(f"OMP relative error:    {error_omp:.4f}")
print(f"\nLASSO identified {np.sum(x_lasso != 0)} nonzero entries (vs true {k})")
print(f"OMP identified {np.sum(x_omp != 0)} nonzero entries (vs true {k})")

## Part 3: Phase Transition


In [ ]:
# Phase transition: k vs m (Donoho-Tanner)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: True signal
ax = axes[0, 0]
ax.stem(range(p), x_true, basefmt=' ')
ax.set_xlabel('Index')
ax.set_ylabel('Value')
ax.set_title(f'True Sparse Signal (k={k})')
ax.set_xlim([0, 100])
ax.grid(True, alpha=0.3)

# Plot 2: Recovered via LASSO
ax = axes[0, 1]
ax.stem(range(p), x_lasso, basefmt=' ', linefmt='C1-', markerfmt='C1o')
ax.stem(range(p), x_true, basefmt=' ', linefmt='C0--', markerfmt='C0x')
ax.set_xlabel('Index')
ax.set_ylabel('Value')
ax.set_title(f'LASSO Recovery (error={error_lasso:.4f})')
ax.set_xlim([0, 100])
ax.legend(['Recovered', 'True'], loc='upper right')
ax.grid(True, alpha=0.3)

# Plot 3: Comparison of methods
ax = axes[1, 0]
methods = ['LASSO', 'OMP']
errors = [error_lasso, error_omp]
colors = ['steelblue', 'orange']
bars = ax.bar(methods, errors, color=colors, alpha=0.7, edgecolor='black')
ax.set_ylabel('Relative Recovery Error')
ax.set_title('LASSO vs OMP Performance')
ax.set_ylim([0, max(errors) * 1.2])
for bar, err in zip(bars, errors):
    ax.text(bar.get_x() + bar.get_width()/2, err, f'{err:.4f}',
           ha='center', va='bottom')
ax.grid(True, alpha=0.3, axis='y')

# Plot 4: Measurement vs Sparsity Tradeoff
ax = axes[1, 1]
p_values = [100, 200, 500, 1000, 2000]
k_values = range(1, 8)
errors_vs_k = []

for k_test in k_values:
    x_sparse = np.zeros(p_values[2])  # Use p=500
    x_sparse[:k_test] = np.random.randn(k_test)
    
    m_test = int(k_test * np.log(p_values[2] / k_test))
    A_test = np.random.randn(m_test, p_values[2]) / np.sqrt(m_test)
    y_test = A_test @ x_sparse + 0.01 * np.random.randn(m_test)
    
    lasso_test = LassoCV(cv=3, max_iter=500)
    lasso_test.fit(A_test, y_test)
    err = np.linalg.norm(lasso_test.coef_ - x_sparse) / (np.linalg.norm(x_sparse) + 1e-10)
    errors_vs_k.append(err)

ax.plot(k_values, errors_vs_k, 'o-', linewidth=2, markersize=8, color='steelblue')
ax.set_xlabel('Sparsity (k)')
ax.set_ylabel('Recovery Error')
ax.set_title('Recovery Error vs Sparsity (m = k*log(p/k))')
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

plt.tight_layout()
plt.savefig('SECTION_2_DATA_SCIENCE/compressed_sensing.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 4: Measurement Bounds


In [ ]:
# Information-theoretic vs practical bounds
fig, ax = plt.subplots(figsize=(10, 6))

p_range = np.logspace(1, 4, 20).astype(int)
k = 10

info_theoretic = k  # Lower bound: entropy of support
comressed_sensing = k * np.log(p_range / k)  # CS bound
naive_bound = k + 100  # Rough naive bound

ax.loglog(p_range, info_theoretic * np.ones_like(p_range), 'g-', linewidth=2, label='Information-Theoretic (Ω(k))')
ax.loglog(p_range, compressed_sensing, 'b-', linewidth=2, label=f'Compressed Sensing (O(k log(p/k)))')
ax.loglog(p_range, naive_bound * np.ones_like(p_range), 'r--', linewidth=2, label='Naive (O(p))')

ax.fill_between(p_range, info_theoretic, compressed_sensing, alpha=0.2, color='blue')
ax.set_xlabel('Signal Dimension (p)', fontsize=12)
ax.set_ylabel('Number of Measurements (m)', fontsize=12)
ax.set_title('Compressed Sensing: The Phase Transition (k=10)', fontsize=12, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('SECTION_2_DATA_SCIENCE/measurement_bounds.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nMEASUREMENT COMPLEXITY")
print("="*60)
print(f"Signal dimension: p")
print(f"Sparsity: k = {k}")
print(f"\n{'p':>8} {'Info Theory':>15} {'CS Bound':>15} {'Savings':>15}")
print("-"*60)
for p_val in [100, 500, 1000, 5000, 10000]:
    cs_bound = k * np.log(p_val / k)
    savings = (1 - cs_bound / p_val) * 100
    print(f"{p_val:>8} {k:>15} {cs_bound:>15.0f} {savings:>14.1f}%")

## Key Takeaways

1. **Sparsity enables underdetermined systems**: With k << p, we can solve y = Ax with m << p
2. **RIP is sufficient**: If A satisfies RIP, LASSO provably recovers sparse signals
3. **Random works**: Gaussian & Bernoulli matrices satisfy RIP with high probability
4. **Phase transition**: Donoho-Tanner curve shows recovery probability as function of m/k
5. **Practical impact**: MRI, radar, signal processing, scientific imaging

### References
- Candès, E. J., & Tao, T. (2005). "Decoding by Linear Programming"
- Donoho, D. L., & Tanner, J. (2005). "Sparse nonnegative solution of underdetermined linear equations by linear programming"
